# 03 - DSP Filter Ablation on Noise-Category Slices

Logical continuation of `02_noise_EDA.ipynb`. There we built a per-segment table
from the markup and listened to segments grouped by `quality` (noise type) and
`noise_level`. Here we **apply our conventional DSP filters** to those same
slices and listen / look at the effect, before vs after.

**Hypothesis (from the project brief):** the conventional filters should reduce
**electric (power-line) noise** via the notch, and the bandpass high-pass edge
should attenuate low-frequency **heartbeat** energy that sits below the
respiratory band. Other noise types (talk, crying, movement, cough) are
broadband / overlap the respiratory band, so we expect the DSP filters to help
them little — that's the gap the learned methods (Demucs etc.) are meant to fill.

The filters tested are the exact `src.model` modules used by `inference.py`
(`NotchFilter`, `BandpassFilter`, `WienerFilter`, `DSPPipeline`) — we import the
classes directly and call them on segment tensors, which keeps this interactive
and segment-native while testing the same code.

**Sections**
- **A.** Filters x noise category (the hypothesis test)
- **B.** Parameter sweeps on representative segments (tuning the defaults)

## 0. Setup

Reuses 02's segment-table loader and `read_segment` (copied here so this
notebook stands alone), and imports the filter modules from `src`.

In [ ]:
import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, HTML, display

# make `import src...` work when running from notebooks/
REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.model import BandpassFilter, DSPPipeline, NotchFilter, WienerFilter

DATA_ROOT = REPO_ROOT / 'data'
MR_DIR = DATA_ROOT / 'medical_records'
MR_WAVS = MR_DIR / 'ALL'
MR_JSON = MR_DIR / 'merged-1761225412230.json'

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)

RANDOM_STATE = 7

In [ ]:
# --- segment table + slicing logic (copied verbatim from 02 so slices match) ---
def has_value(x):
    return x is not None and x != '' and not (isinstance(x, float) and math.isnan(x))


WEAK_NOISE_QUALITIES = {'electric noise', 'quiet sound'}
STRONG_NOISE_QUALITIES = {
    'talk', 'crying', 'cough', 'movement', 'heart', 'other', 'unidentified', 'clothes'
}


def is_broad_noisy(row):
    quality = row.get('quality')
    noise_level = row.get('noise_level')
    return (has_value(quality) and quality != 'clean') or (has_value(noise_level) and noise_level != 'low')


def is_strong_noisy(row):
    quality = row.get('quality')
    noise_level = row.get('noise_level')
    return noise_level in {'moderate', 'high'} or quality in STRONG_NOISE_QUALITIES


def load_segment_dataframe(json_path=MR_JSON, wav_dir=MR_WAVS):
    with open(json_path, encoding='utf-8-sig') as f:
        payload = json.load(f)
    rows = []
    for file_idx, file_entry in enumerate(payload['files']):
        filename = file_entry['filename']
        wav_path = wav_dir / filename
        for markup_idx, segment in enumerate(file_entry.get('markup', [])):
            row = {
                'segment_id': f'{filename}:{markup_idx}',
                'uid': file_entry.get('uid'),
                'doctor': file_entry.get('doctor'),
                'filename': filename,
                'wav_path': str(wav_path),
                'wav_exists': wav_path.exists(),
                'start': float(segment.get('start', 0.0) or 0.0),
                'length': float(segment.get('length', 0.0) or 0.0),
                'phase': segment.get('phase'),
                'pathology': segment.get('pathology'),
                'quality': segment.get('quality'),
                'noise_level': segment.get('noise_level'),
            }
            row['end'] = row['start'] + row['length']
            row['has_pathology'] = has_value(row['pathology'])
            row['broad_noisy'] = is_broad_noisy(row)
            row['strong_noisy'] = is_strong_noisy(row)
            row['clean_target'] = row['has_pathology'] and not row['broad_noisy']
            row['noise_only'] = (not row['has_pathology']) and row['broad_noisy']
            rows.append(row)
    return pd.DataFrame(rows)


segments = load_segment_dataframe()
segments = segments.loc[segments['wav_exists']].reset_index(drop=True)
print(f'{len(segments)} segments from {segments.filename.nunique()} files')
segments.head()

In [ ]:
# --- audio reading (from 02) ---
def read_segment(row, pad_sec=0.05, normalize=False):
    """Read one annotated segment from disk -> (audio float32 mono, sample_rate)."""
    wav_path = Path(row['wav_path'])
    info = sf.info(str(wav_path))
    sr = info.samplerate
    start_sec = max(0.0, float(row['start']) - pad_sec)
    end_sec = min(float(info.frames) / sr, float(row['end']) + pad_sec)
    start_frame = int(round(start_sec * sr))
    frames = max(1, int(round((end_sec - start_sec) * sr)))
    audio, sr = sf.read(str(wav_path), start=start_frame, frames=frames, dtype='float32', always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if normalize:
        peak = np.max(np.abs(audio)) if len(audio) else 0.0
        if peak > 0:
            audio = audio / peak * 0.95
    return audio, sr

In [ ]:
# --- filter application: same modules inference.py uses ---
def apply_filter(audio, sr, filter_module):
    """Run a src.model filter on a 1-D numpy segment, return 1-D numpy.

    Filters follow the contract forward(audio[B,1,T]) -> {'audio': [B,1,T]}.
    """
    x = torch.from_numpy(np.asarray(audio, dtype=np.float32))[None, None, :]
    with torch.no_grad():
        out = filter_module(audio=x)['audio']
    return out[0, 0].cpu().numpy()


def build_filters(sr):
    """Instantiate the filter bank at a given native sample rate.

    Defaults mirror the Hydra configs (model/notch.yaml, bandpass.yaml,
    wiener.yaml, dsp_baseline*.yaml).
    """
    notch = NotchFilter(sample_rate=sr, freq=50.0, n_harmonics=4, quality=30.0)
    bandpass = BandpassFilter(sample_rate=sr, low_hz=100.0, high_hz=2000.0, order=4)
    wiener = WienerFilter(sample_rate=sr, n_fft=1024, hop_length=None, noise_percentile=10.0, gain_floor=0.05)
    notch_bp = DSPPipeline([
        NotchFilter(sample_rate=sr, freq=50.0, n_harmonics=4, quality=30.0),
        BandpassFilter(sample_rate=sr, low_hz=100.0, high_hz=2000.0, order=4),
    ])
    full = DSPPipeline([
        NotchFilter(sample_rate=sr, freq=50.0, n_harmonics=4, quality=30.0),
        BandpassFilter(sample_rate=sr, low_hz=100.0, high_hz=2000.0, order=4),
        WienerFilter(sample_rate=sr, n_fft=1024, noise_percentile=10.0, gain_floor=0.05),
    ])
    # order matters for display: original first, then increasing processing
    return {
        'original': None,
        'notch': notch,
        'bandpass': bandpass,
        'notch+bandpass': notch_bp,
        'wiener': wiener,
        'full (n+bp+w)': full,
    }

In [ ]:
# --- display helpers: players + side-by-side spectrograms ---
def _norm(x):
    peak = np.max(np.abs(x)) if len(x) else 0.0
    return x / peak * 0.95 if peak > 0 else x


def compare_filters(row, variants=None, fmax=4000, normalize_audio=True):
    """For one segment: top row = spectrogram of original vs each filtered
    variant; bottom row = spectrogram of the RESIDUAL (filtered - original),
    titled with how much each filter actually changed the signal. Players are
    shown two-per-row: filtered output (left) and its residual (right).
    """
    audio, sr = read_segment(row, pad_sec=0.05, normalize=False)
    bank = build_filters(sr)
    if variants is not None:
        bank = {k: bank[k] for k in variants if k in bank}

    label = (f"{row['segment_id']} | quality={row.get('quality')} | "
             f"level={row.get('noise_level')} | pathology={row.get('pathology')} | "
             f"{row['length']:.2f}s | sr={sr}")
    display(HTML(f'<b>{label}</b>'))

    in_rms = float(np.sqrt(np.mean(audio ** 2))) + 1e-12

    n = len(bank)
    fig, axes = plt.subplots(2, n, figsize=(3.0 * n, 5.6), squeeze=False)
    rendered = {}   # name -> filtered signal
    residuals = {}  # name -> residual signal (None for identity/original)
    for col, (name, filt) in enumerate(bank.items()):
        y = audio if filt is None else apply_filter(audio, sr, filt)
        m = min(len(y), len(audio))
        y, ref = y[:m], audio[:m]
        rendered[name] = y

        ax_top = axes[0, col]
        ax_top.specgram(y, Fs=sr, NFFT=512, noverlap=384, cmap='magma')
        ax_top.set_title(name, fontsize=9)
        ax_top.set_ylim(0, min(fmax, sr // 2))
        ax_top.label_outer()

        resid = y - ref
        resid_rms = float(np.sqrt(np.mean(resid ** 2)))
        resid_db = 20.0 * np.log10(resid_rms / in_rms + 1e-12)
        ax_bot = axes[1, col]
        if name == 'original' or resid_rms == 0.0:
            ax_bot.text(0.5, 0.5, 'identity\n(Δ = 0)', ha='center', va='center',
                        transform=ax_bot.transAxes, fontsize=9)
            ax_bot.set_xticks([]); ax_bot.set_yticks([])
            residuals[name] = None
        else:
            ax_bot.specgram(resid, Fs=sr, NFFT=512, noverlap=384, cmap='magma')
            ax_bot.set_ylim(0, min(fmax, sr // 2))
            ax_bot.label_outer()
            residuals[name] = resid
        ax_bot.set_title(f"Δ rms={resid_rms:.2e} ({resid_db:+.1f} dB)", fontsize=8)
        ax_bot.set_xlabel('s', fontsize=8)

    plt.tight_layout()
    plt.show()

    # players: two columns -> output (left) | residual (right)
    def _player_html(x):
        if x is None:
            return '<i style="color:gray">—</i>'
        y = _norm(x) if normalize_audio else x
        return Audio(y, rate=sr)._repr_html_()

    rows_html = [
        '<table style="width:100%; text-align:left">'
        '<tr><th>filter</th><th>output</th><th>residual (out − orig)</th></tr>'
    ]
    for name, y in rendered.items():
        rows_html.append(
            f'<tr><td><b>{name}</b></td>'
            f'<td>{_player_html(y)}</td>'
            f'<td>{_player_html(residuals[name])}</td></tr>'
        )
    rows_html.append('</table>')
    display(HTML(''.join(rows_html)))




def sample_category(quality=None, noise_level=None, with_pathology=None, n=3, random_state=RANDOM_STATE):
    mask = pd.Series(True, index=segments.index)
    if quality is not None:
        mask &= segments['quality'] == quality
    if noise_level is not None:
        mask &= segments['noise_level'] == noise_level
    if with_pathology is not None:
        mask &= segments['has_pathology'] == with_pathology
    part = segments.loc[mask]
    if len(part) == 0:
        return part
    return part.sample(min(n, len(part)), random_state=random_state)

## A. Filters x noise category

The core hypothesis test. For each noise category we sample a few segments and
compare original vs each filter variant (spectrogram + audio).

We lead with the two categories the DSP filters are *supposed* to help:
**electric noise** (notch) and **heart** (bandpass high-pass edge). Then the
broadband categories where we expect little DSP benefit.

### A.1 Electric noise -- the notch's target

Watch for a horizontal line near 50 Hz (and harmonics: 100, 150, ... Hz) in the
`original` spectrogram that disappears in `notch` / `notch+bandpass`.

In [ ]:
for _, row in sample_category(quality='electric noise', n=3, random_state=RANDOM_STATE + 1).iterrows():
    compare_filters(row)

### A.2 Heart -- the bandpass high-pass target

Heart sounds concentrate low (< ~150 Hz). The bandpass `low_hz=100` edge should
attenuate them. Watch the low-frequency band of the spectrogram.

In [ ]:
for _, row in sample_category(quality='heart', n=3, random_state=RANDOM_STATE + 1).iterrows():
    compare_filters(row)

### A.3 Broadband / in-band noise -- expected to be hard for DSP

talk / crying / movement / cough overlap the respiratory band, so notch and
bandpass can't separate them; wiener may shave some stationary part. This is the
gap we expect learned methods to fill -- documenting the DSP ceiling here.

In [ ]:
for quality in ['talk', 'movement', 'cough', 'crying']:
    part = sample_category(quality=quality, n=2, random_state=RANDOM_STATE + 1)
    if len(part) == 0:
        print(f'(no segments for quality={quality})')
        continue
    display(HTML(f'<h4>quality = {quality}</h4>'))
    for _, row in part.iterrows():
        compare_filters(row)

### A.4 By noise level

Same comparison sliced by `noise_level` instead of type, to see whether the DSP
benefit changes with severity.

In [ ]:
for level in ['low', 'moderate', 'high']:
    part = sample_category(noise_level=level, n=2, random_state=RANDOM_STATE + 1)
    if len(part) == 0:
        print(f'(no segments for noise_level={level})')
        continue
    display(HTML(f'<h4>noise_level = {level}</h4>'))
    for _, row in part.iterrows():
        compare_filters(row)

### A.5 Does the filtering harm clean pathology segments?

Important control: the filters must NOT destroy diagnostic content on segments
that are already clean and carry a pathology label. If bandpass/wiener smear the
wheezes/crepitus here, that's a cost to weigh.

In [ ]:
clean_path = segments.loc[segments['clean_target'] & segments['pathology'].notna()]
for _, row in clean_path.sample(min(3, len(clean_path)), random_state=RANDOM_STATE).iterrows():
    compare_filters(row)

## B. Parameter sweeps on representative segments

Tuning the defaults. Pick a representative segment per knob and sweep, listening
to the effect. Override the chosen values back in the Hydra configs once decided.

In [ ]:
def sweep(row, variants, fmax=4000, normalize_audio=True):
    """Param-sweep view: variants is dict[name -> filter_module]; 'original'
    prepended. Top row = spectrogram of each variant; bottom row = spectrogram
    of the RESIDUAL (variant - original) with Δ rms / dB. Players shown
    two-per-row: output (left) | residual (right).
    """
    audio, sr = read_segment(row, pad_sec=0.05, normalize=False)
    items = [('original', None)] + list(variants.items())
    label = (f"{row['segment_id']} | quality={row.get('quality')} | "
             f"level={row.get('noise_level')} | sr={sr}")
    display(HTML(f'<b>{label}</b>'))

    in_rms = float(np.sqrt(np.mean(audio ** 2))) + 1e-12

    n = len(items)
    fig, axes = plt.subplots(2, n, figsize=(3.0 * n, 5.6), squeeze=False)
    rendered = {}   # name -> variant signal
    residuals = {}  # name -> residual signal (None for identity/original)
    for col, (name, filt) in enumerate(items):
        y = audio if filt is None else apply_filter(audio, sr, filt)
        m = min(len(y), len(audio))
        y, ref = y[:m], audio[:m]
        rendered[name] = y

        ax_top = axes[0, col]
        ax_top.specgram(y, Fs=sr, NFFT=512, noverlap=384, cmap='magma')
        ax_top.set_title(name, fontsize=9)
        ax_top.set_ylim(0, min(fmax, sr // 2))
        ax_top.label_outer()

        resid = y - ref
        resid_rms = float(np.sqrt(np.mean(resid ** 2)))
        resid_db = 20.0 * np.log10(resid_rms / in_rms + 1e-12)
        ax_bot = axes[1, col]
        if name == 'original' or resid_rms == 0.0:
            ax_bot.text(0.5, 0.5, 'identity\n(Δ = 0)', ha='center', va='center',
                        transform=ax_bot.transAxes, fontsize=9)
            ax_bot.set_xticks([]); ax_bot.set_yticks([])
            residuals[name] = None
        else:
            ax_bot.specgram(resid, Fs=sr, NFFT=512, noverlap=384, cmap='magma')
            ax_bot.set_ylim(0, min(fmax, sr // 2))
            ax_bot.label_outer()
            residuals[name] = resid
        ax_bot.set_title(f"Δ rms={resid_rms:.2e} ({resid_db:+.1f} dB)", fontsize=8)
        ax_bot.set_xlabel('s', fontsize=8)

    plt.tight_layout()
    plt.show()

    # players: two columns -> output (left) | residual (right)
    def _player_html(x):
        if x is None:
            return '<i style="color:gray">—</i>'
        y = _norm(x) if normalize_audio else x
        return Audio(y, rate=sr)._repr_html_()

    rows_html = [
        '<table style="width:100%; text-align:left">'
        '<tr><th>variant</th><th>output</th><th>residual (out − orig)</th></tr>'
    ]
    for name, y in rendered.items():
        rows_html.append(
            f'<tr><td><b>{name}</b></td>'
            f'<td>{_player_html(y)}</td>'
            f'<td>{_player_html(residuals[name])}</td></tr>'
        )
    rows_html.append('</table>')
    display(HTML(''.join(rows_html)))


### B.1 Notch Q (width) on an electric-noise segment

Higher Q = narrower notch (removes less around 50 Hz). Find the Q that kills the
hum without gouging a hole in nearby content.

In [ ]:
elec = sample_category(quality='electric noise', n=1, random_state=RANDOM_STATE + 2)
if len(elec):
    row = elec.iloc[0]
    sr = sf.info(row['wav_path']).samplerate
    sweep(row, {f'Q={q}': NotchFilter(sample_rate=sr, freq=50.0, n_harmonics=4, quality=q)
                for q in [10, 30, 60]})
else:
    print('no electric-noise segment found')

### B.2 Bandpass cutoffs

Sweep the low/high edges. Lower `low_hz` keeps more heartbeat; higher trims it
but risks low breath energy. 2000 vs 1000 Hz top edge affects high-freq wheeze.

In [ ]:
bp_row = sample_category(quality='heart', n=1, random_state=RANDOM_STATE)
if len(bp_row):
    row = bp_row.iloc[0]
    sr = sf.info(row['wav_path']).samplerate
    sweep(row, {
        '50-2000': BandpassFilter(sample_rate=sr, low_hz=50.0, high_hz=2000.0),
        '100-2000': BandpassFilter(sample_rate=sr, low_hz=100.0, high_hz=2000.0),
        '150-1500': BandpassFilter(sample_rate=sr, low_hz=150.0, high_hz=1500.0),
    })
else:
    print('no heart segment found')

### B.3 Wiener noise_percentile and gain_floor

`noise_percentile` higher = assume more is noise (more aggressive).
`gain_floor` lower = harder suppression (more musical-noise warble).

In [ ]:
w_row = sample_category(noise_level='moderate', n=1, random_state=RANDOM_STATE + 8)
if len(w_row):
    row = w_row.iloc[0]
    sr = sf.info(row['wav_path']).samplerate
    sweep(row, {
        'pct=5,gf=.05': WienerFilter(sample_rate=sr, noise_percentile=5.0, gain_floor=0.05),
        'pct=10,gf=.05': WienerFilter(sample_rate=sr, noise_percentile=10.0, gain_floor=0.05),
        'pct=20,gf=.0': WienerFilter(sample_rate=sr, noise_percentile=20.0, gain_floor=0.0),
    })
else:
    print('no moderate-noise segment found')

## Notes / takeaways

_Fill in after listening:_
- Electric noise: notch effective? Q to use?
- Heart: bandpass low edge enough, or need higher `low_hz`?
- Broadband (talk/movement/...): DSP ceiling confirmed? -> motivates learned methods.
- Clean pathology control: any diagnostic content harmed?
- Chosen tuned defaults to push back into the Hydra configs.